# Eadro RCA — Publication-Safe, SocialNetwork-Only (Stage 1)

Run this top-to-bottom in Google Colab (Runtime > Run all is fine — every step is resumable/idempotent).

This is the **first, fast stage**: SocialNetwork (SN) only, all 5 configurations, 3 seeds each,
with a proper train/validation/test protocol (no test-set leakage in checkpoint selection).

Once this finishes and you've checked the results look sane, **Stage 2** (see the last markdown
cell) is just: download + preprocess TrainTicket (TT), flip one line back to
`DATASETS = ["SN", "TT"]`, and re-run the training cell. It will automatically skip every SN run
that's already finished and only train the new TT runs — nothing here needs to be rebuilt.

Configurations: Original Eadro (baseline), LSTM + Temporal Attention (α = 0.5 / 0.6 / 0.7),
and Hybrid DCC+LSTM (α = 0.5).


# Eadro RCA — SN data-only (for the live-inference demo)

This is a trimmed copy of `Eadro_SN_only_publication_safe_2.ipynb`: it runs **environment setup -> SN download -> preprocessing only**. It does **not** retrain anything (your real trained checkpoints already exist). The only goal here is to regenerate `chunk_test.pkl` and `metadata.json` for SN so they can be wired into the Streamlit demo for genuine live-model inference.

Run every cell top to bottom. Takes a few minutes, not hours -- there is no training in this notebook.

## 1. Environment and repository

In [ ]:
%cd /content

!test -x /content/bin/micromamba || (wget -qO- https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba)
!test -x /content/eadro-env/bin/python || /content/bin/micromamba create -y -p /content/eadro-env python=3.8 pip

!/content/eadro-env/bin/pip install -q \
    numpy==1.21.5 \
    pandas==1.4.2 \
    scikit-learn==1.1.2 \
    torch==1.12.1 \
    dgl==0.9.1 \
    tick==0.7.0.1 \
    drain3 \
    matplotlib

!test -d /content/Eadro-full || git clone https://github.com/FeiGSSS/Eadro.git /content/Eadro-full

!/content/eadro-env/bin/python -c "import torch,dgl,numpy,pandas,sklearn,tick,drain3; print('Torch',torch.__version__,'| DGL',dgl.__version__,'| environment OK')"


/content
bin/micromamba
[+] 0.0s
[+] 0.1s
Fetch Shard Index for conda-forge/linux-64                                                ✔ Done (0.1 sec)
Fetch Shard Index for conda-forge/noarch                                                  ✔ Done (0.1 sec)
Fetching and Parsing Packages' Shards                                                     ✔ Done (6.8 sec)

Resolving Environment                                                                     ✔ Done (0.2 sec)

Transaction

  Prefix: /content/eadro-env

  Updating specs:

   - python=3.8
   - pip


  Package               Version  Build                 Channel          Size
──────────────────────────────────────────────────────────────────────────────
  Install:
──────────────────────────────────────────────────────────────────────────────

  + _openmp_mutex           4.5  20_gnu                conda-forge      29kB
  + bzip2                 1.0.8  hda65f42_10           conda-forge     258kB
  + ca-certificates   2026.7.22  hbd8a

## 2. Download the SocialNetwork (SN) dataset only

In [ ]:
%cd /content
!mkdir -p /content/eadro-data
%cd /content/eadro-data

!test -f "SN Dataset.zip" || wget -q --show-progress -O "SN Dataset.zip" "https://zenodo.org/records/7615394/files/SN%20Dataset.zip?download=1"
!test -d "SN Dataset" || unzip -q "SN Dataset.zip"

!find /content/eadro-data -maxdepth 2 -type d | sort


/content
/content/eadro-data
SN Dataset.zip      100%[===================>]  75.11M  2.84MB/s    in 56s     
/content/eadro-data
/content/eadro-data/__MACOSX
/content/eadro-data/__MACOSX/SN Dataset
/content/eadro-data/SN Dataset
/content/eadro-data/SN Dataset/data
/content/eadro-data/SN Dataset/no fault


## 3. Preprocess SN and build the deterministic original train/test split

**Important fix applied here:** the raw parser scripts (`parse_raw_records.py`, `parse_raw_metrics.py`,
`parse_raw_traces.py`, `parse_raw_logs.py`) hard-code `for dataset_name in ['SN', 'TT']:` in the
original repo. Since we have not downloaded TT in this stage, that loop would crash trying to open a
`TT Dataset` folder that doesn't exist. The `sed` commands below patch each script down to
`for dataset_name in ['SN']:` before they run. When you add TT in Stage 2, re-clone a fresh copy of
the repo (or revert these `sed` patches) so both datasets get processed again.


In [ ]:
%env ROOT_PATH=/content/eadro-data
%cd /content/Eadro-full/codes/preprocess

# Fork compatibility fix (column name mismatch between the parser and single_process.py).
!sed -i 's/groupby("event")/groupby("events")/' /content/Eadro-full/codes/preprocess/single_process.py

# SN-only fix: these four scripts default to looping over ['SN', 'TT'].
# TT is not downloaded in this stage, so restrict them to SN only.
!sed -i "s/for dataset_name in \['SN', 'TT'\]:/for dataset_name in ['SN']:/" parse_raw_records.py
!sed -i "s/for dataset_name in \['SN', 'TT'\]:/for dataset_name in ['SN']:/" parse_raw_metrics.py
!sed -i "s/for dataset_name in \['SN', 'TT'\]:/for dataset_name in ['SN']:/" parse_raw_traces.py
!sed -i "s/for dataset_name in \['SN', 'TT'\]:/for dataset_name in ['SN']:/" parse_raw_logs.py

!grep -n "for dataset_name in" parse_raw_records.py parse_raw_metrics.py parse_raw_traces.py parse_raw_logs.py

# Deterministic preprocessing split.
!grep -q "EADRO_REPRO_SEED" align.py || sed -i '/import random/a random.seed(2026)  # EADRO_REPRO_SEED' align.py
!grep -q "EADRO_NUMPY_REPRO_SEED" align.py || sed -i '/import numpy as np/a np.random.seed(2026)  # EADRO_NUMPY_REPRO_SEED' align.py

# Raw -> parsed_data. Skip expensive stages when outputs already exist (safe to re-run this cell).
!test -f parsed_data/SN/records0.json || /content/eadro-env/bin/python parse_raw_records.py
!test -d parsed_data/SN/metrics0 || /content/eadro-env/bin/python parse_raw_metrics.py
!test -f parsed_data/SN/traces0.json || /content/eadro-env/bin/python parse_raw_traces.py
!test -f parsed_data/SN/templates.json || (/content/eadro-env/bin/python parse_raw_logs.py > /content/parse_raw_logs_output.txt 2>&1)

# Rebuild only the final split files using the deterministic seed above.
# NOTE: chunks.pkl (the intermediate, pre-split file) also has to be removed
# here, not just chunk_train.pkl/chunk_test.pkl. If chunks.pkl already exists,
# align.py's split_chunks() takes a shortcut branch that has a real bug in
# the upstream Eadro repo (it calls chunks.update(...) on a variable that was
# never initialized on that path) -- it crashes with a NameError, but since
# this is a plain `!python ...` call the notebook doesn't stop, it just
# silently fails to (re)write chunk_train.pkl/chunk_test.pkl. Deleting
# chunks.pkl too forces the clean, non-buggy regeneration path every time.
!rm -f /content/Eadro-full/codes/chunks/SN/chunks.pkl
!rm -f /content/Eadro-full/codes/chunks/SN/chunk_train.pkl /content/Eadro-full/codes/chunks/SN/chunk_test.pkl

!/content/eadro-env/bin/python align.py --name SN --chunk_lenth 10 --test_ratio 0.3

!ls -lh /content/Eadro-full/codes/chunks/SN


env: ROOT_PATH=/content/eadro-data
/content/Eadro-full/codes/preprocess
parse_raw_records.py:83:    for dataset_name in ['SN']:
parse_raw_metrics.py:22:    for dataset_name in ['SN']:
parse_raw_traces.py:50:    for dataset_name in ['SN']:
parse_raw_logs.py:90:    for dataset_name in ['SN']:
# Node num: 12


 ^^^^^^^^^^^^^^^^^^^^ Now dealing with 0 ^^^^^^^^^^^^^^^^^^^^
# starts at 1650219165/1650219165 and ends at 1650220575/1650220576
*** Dealing with traces...
*** Dealing with metrics...
*** Dealing with logs...
# Real Template Number: 0
Computing Hawkes of chunk 100/1402
Computing Hawkes of chunk 200/1402
Computing Hawkes of chunk 300/1402
Computing Hawkes of chunk 400/1402
Computing Hawkes of chunk 500/1402
Computing Hawkes of chunk 600/1402
Computing Hawkes of chunk 700/1402
Computing Hawkes of chunk 800/1402
Computing Hawkes of chunk 900/1402
Computing Hawkes of chunk 1000/1402
Computing Hawkes of chunk 1100/1402
Computing Hawkes of chunk 1200/1402
Computing Hawkes of chunk 1300/1

In [ ]:
import shutil
from pathlib import Path
from google.colab import files, drive

CHUNK_DIR = Path("/content/Eadro-full/codes/chunks/SN")
assert (CHUNK_DIR / "chunk_test.pkl").exists(), "chunk_test.pkl missing -- re-run section 3 above"
assert (CHUNK_DIR / "metadata.json").exists(), "metadata.json missing -- re-run section 3 above"

# Package just the two files the live-inference demo needs (small, no need to
# ship chunk_train.pkl too).
OUT_DIR = Path("/content/sn_chunk_data")
OUT_DIR.mkdir(exist_ok=True)
shutil.copy(CHUNK_DIR / "chunk_test.pkl", OUT_DIR / "chunk_test.pkl")
shutil.copy(CHUNK_DIR / "metadata.json", OUT_DIR / "metadata.json")

zip_path = shutil.make_archive("/content/sn_chunk_data", "zip", str(OUT_DIR))
print(f"Packaged: {zip_path}")
print(f"  chunk_test.pkl : {(OUT_DIR / 'chunk_test.pkl').stat().st_size / 1e6:.1f} MB")
print(f"  metadata.json  : {(OUT_DIR / 'metadata.json').stat().st_size} bytes")

# Also back it up to Drive, same pattern as the other notebooks, in case the
# direct browser download below gets interrupted.
try:
    drive.mount("/content/drive", force_remount=False)
    drive_dir = Path("/content/drive/MyDrive/eadro-paper")
    drive_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy(zip_path, drive_dir / "sn_chunk_data.zip")
    print(f"Also backed up to {drive_dir / 'sn_chunk_data.zip'}")
except Exception as e:
    print(f"Drive backup skipped ({e}) -- direct download below still works")

# Direct browser download -- this is the fastest path, use this.
files.download(zip_path)
print("\nUpload sn_chunk_data.zip to Claude once the download finishes.")